# BirdCLEF+ 2026 Perch v2 Training

Train the PyTorch probe from cached Google Perch v2 embeddings and write the artifact used by `04_perch_v2_submit.ipynb`. This notebook does not load the Perch TensorFlow SavedModel; embedding extraction is treated as an upstream artifact step.


## 1. Setup

Load metadata, configure artifact paths, and keep the runtime focused on cached embeddings plus PyTorch training.


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    """Configuration for the cached-embedding Perch probe training notebook."""

    seed = 42
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    aligned_embeddings_name = "train_embeddings_aligned.npz"
    attached_artifact_root = Path(
        "/kaggle/input/models/tuannm3812/"
        "birdclef-perch-v2-artifacts/pytorch/default"
    )
    perch_meta_dir = Path("/kaggle/input/datasets/jaejohn/perch-meta")
    full_embeddings_path = perch_meta_dir / "full_perch_arrays.npz"
    full_metadata_path = perch_meta_dir / "full_perch_meta.parquet"
    legacy_embeddings_path = attached_artifact_root / "1/perch_v2/train_embeddings.npz"
    embedding_dim = 1536
    probe_batch_size = 128
    probe_epochs = 20
    early_stopping_patience = 4
    early_stopping_min_delta = 1e-4
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    temperature_grid = np.linspace(0.6, 2.5, 39)
    prior_clip = 1.5
    prior_smoothing = 0.5


def seed_everything(seed: int = 42) -> None:
    """Set deterministic seeds for Python and NumPy."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def validate_data_root(path: Path) -> Path:
    """Validate the fixed BirdCLEF+ 2026 Kaggle competition input."""
    if (path / "train.csv").exists() or (path / "sample_submission.csv").exists():
        return path
    raise FileNotFoundError(f"BirdCLEF 2026 competition data not found at {path}.")

seed_everything(CFG.seed)
CFG.data_root = validate_data_root(CFG.data_root)
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Artifact directory: {CFG.artifact_dir}")

from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None

    def display(obj: object) -> None:
        print(obj)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
print(f"Device: {device}")


## 2. Metadata And Cached Embeddings

Load the competition metadata, label order, taxonomy context, and cached Perch embeddings. The cached embedding artifact is the only external model artifact needed for this training notebook.


In [ ]:
def find_own_aligned_embedding_paths() -> list[Path]:
    """Find aligned embedding caches from previously uploaded artifact versions."""
    if not CFG.attached_artifact_root.exists():
        return []
    return sorted(
        CFG.attached_artifact_root.glob(f"*/perch_v2/{CFG.aligned_embeddings_name}"),
        key=lambda path: int(path.parts[-3]) if path.parts[-3].isdigit() else -1,
        reverse=True,
    )


def find_train_embedding_candidates() -> list[Path]:
    """Find cached Perch embedding inputs in preferred fallback order."""
    ordered = [
        *find_own_aligned_embedding_paths(),
        CFG.legacy_embeddings_path,
        CFG.full_embeddings_path,
    ]
    candidates = []
    seen = set()
    for path in ordered:
        if path in seen:
            continue
        seen.add(path)
        if path.exists():
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            "Could not find cached Perch embeddings. Attach a previous aligned "
            "artifact, artifact version 1, or perch-meta."
        )
    return candidates


def npz_embedding_array(saved: np.lib.npyio.NpzFile) -> np.ndarray:
    """Return the first likely 2D embedding array from an npz file."""
    for key in ["embeddings", "features", "x", "arr_0"]:
        if key in saved and saved[key].ndim == 2:
            return saved[key].astype(np.float32)
    for key in saved.files:
        if saved[key].ndim == 2:
            return saved[key].astype(np.float32)
    raise KeyError(f"No 2D embedding array found in npz keys: {saved.files}")


def metadata_path_for_embeddings(path: Path) -> Path | None:
    """Return sidecar metadata for full Perch arrays when available."""
    if path == CFG.full_embeddings_path and CFG.full_metadata_path.exists():
        return CFG.full_metadata_path
    candidate = path.with_name("full_perch_meta.parquet")
    return candidate if candidate.exists() else None


def align_embeddings_with_metadata(
    embeddings: np.ndarray,
    metadata_path: Path,
    train_df: pd.DataFrame,
) -> np.ndarray:
    """Align full Perch arrays to train.csv order using the metadata parquet."""
    meta = pd.read_parquet(metadata_path).reset_index(drop=True)
    if len(meta) != len(embeddings):
        raise ValueError(
            f"Metadata rows {len(meta):,} do not match embedding rows {len(embeddings):,}."
        )
    if "filename" not in meta.columns:
        raise ValueError(f"{metadata_path} must contain a filename column.")

    meta["filename"] = meta["filename"].astype(str)
    train_filenames = train_df["filename"].astype(str).to_numpy()
    groups = meta.groupby("filename", sort=False).indices
    missing = [filename for filename in train_filenames if filename not in groups]
    if missing:
        raise ValueError(f"No cached Perch embedding found for {missing[0]}.")

    duplicate_rows = int(meta["filename"].duplicated().sum())
    if duplicate_rows:
        print(
            f"Found {duplicate_rows:,} extra cached embedding rows for duplicate filenames; "
            "mean-pooling them per train clip."
        )

    aligned = np.empty((len(train_filenames), embeddings.shape[1]), dtype=np.float32)
    for row_idx, filename in enumerate(train_filenames):
        embedding_idx = np.asarray(groups[filename], dtype=np.int64)
        aligned[row_idx] = embeddings[embedding_idx].mean(axis=0)
    return aligned


def load_embedding_cache(path: Path, train_df: pd.DataFrame) -> np.ndarray:
    """Load cached embeddings and align them to train.csv row order."""
    saved = np.load(path, allow_pickle=True)
    embeddings = npz_embedding_array(saved)
    metadata_path = metadata_path_for_embeddings(path)

    if metadata_path is not None:
        return align_embeddings_with_metadata(embeddings, metadata_path, train_df)

    if len(embeddings) != len(train_df):
        raise ValueError(
            f"Embedding row count {len(embeddings):,} does not match train.csv "
            f"row count {len(train_df):,}."
        )
    if "filenames" in saved:
        cached_filenames = saved["filenames"].astype(str)
        train_filenames = train_df["filename"].astype(str).to_numpy()
        if not np.array_equal(cached_filenames, train_filenames):
            order = pd.Series(np.arange(len(cached_filenames)), index=cached_filenames)
            indices = order.reindex(train_filenames)
            if indices.isna().any():
                missing = train_filenames[indices.isna()][0]
                raise ValueError(f"Cached embeddings are missing filename {missing}.")
            embeddings = embeddings[indices.to_numpy(dtype=np.int64)]
    return embeddings


def save_aligned_embedding_cache(embeddings: np.ndarray, train_df: pd.DataFrame) -> Path:
    """Save a project-owned train embedding cache aligned to train.csv."""
    path = CFG.artifact_dir / CFG.aligned_embeddings_name
    np.savez_compressed(
        path,
        embeddings=embeddings.astype(np.float32),
        filenames=train_df["filename"].astype(str).to_numpy(),
        primary_labels=train_df["primary_label"].astype(str).to_numpy(),
    )
    return path


train = pd.read_csv(CFG.data_root / "train.csv")
train["primary_label"] = train["primary_label"].astype(str)

taxonomy_path = CFG.data_root / "taxonomy.csv"
taxonomy = pd.read_csv(taxonomy_path) if taxonomy_path.exists() else pd.DataFrame()
if not taxonomy.empty and "primary_label" in taxonomy.columns:
    taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)

labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)

label_metadata = (
    train.groupby("primary_label")
    .agg(train_support=("filename", "count"))
    .reset_index()
)
if not taxonomy.empty and {"primary_label", "class_name"}.issubset(taxonomy.columns):
    label_metadata = label_metadata.merge(
        taxonomy[["primary_label", "class_name"]].drop_duplicates(),
        on="primary_label",
        how="left",
    )

embedding_errors = []
for embeddings_path in find_train_embedding_candidates():
    try:
        embeddings = load_embedding_cache(embeddings_path, train)
        break
    except (KeyError, ValueError) as error:
        embedding_errors.append(f"{embeddings_path}: {error}")
        print(f"Skipping cached embeddings at {embeddings_path}: {error}")
else:
    raise RuntimeError(
        "No cached Perch embedding input could be aligned to train.csv. "
        + " | ".join(embedding_errors)
    )

aligned_cache_path = save_aligned_embedding_cache(embeddings, train)

if CFG.max_samples:
    sample_idx = train.sample(CFG.max_samples, random_state=CFG.seed).index.to_numpy()
    train = train.iloc[sample_idx].reset_index(drop=True)
    embeddings = embeddings[sample_idx]

(CFG.artifact_dir / "labels.json").write_text(
    json.dumps(idx_to_label, indent=2),
    encoding="utf-8",
)
label_metadata.to_csv(CFG.artifact_dir / "label_metadata.csv", index=False)

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
print(f"Embeddings: {embeddings.shape}")
print(f"Loaded embeddings from: {embeddings_path}")
print(f"Saved aligned cache to: {aligned_cache_path}")
display(train.head())


## 3. Probe Model

Train a shallow classifier on frozen Perch embeddings. Singleton classes stay in training so the stratified validation split remains valid.


In [ ]:
class PerchProbe(nn.Module):
    """Shallow classifier trained on frozen Perch embeddings."""

    def __init__(self, embedding_dim: int, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def torch_load_checkpoint(path: Path) -> dict[str, object]:
    """Load a PyTorch checkpoint with Kaggle-version compatibility."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def safe_train_valid_split(
    targets: np.ndarray,
    test_size: float = 0.2,
) -> tuple[np.ndarray, np.ndarray]:
    """Create a stratified split while keeping singleton classes in train."""
    counts = pd.Series(targets).value_counts()
    rare_classes = set(counts[counts < 2].index)
    all_idx = np.arange(len(targets))
    rare_idx = np.array(
        [idx for idx in all_idx if targets[idx] in rare_classes],
        dtype=np.int64,
    )
    common_idx = np.array(
        [idx for idx in all_idx if targets[idx] not in rare_classes],
        dtype=np.int64,
    )
    train_common, valid_idx = train_test_split(
        common_idx,
        test_size=test_size,
        random_state=CFG.seed,
        stratify=targets[common_idx],
    )
    train_idx = np.concatenate([train_common, rare_idx])
    return train_idx, valid_idx


x = embeddings.astype(np.float32)
y = train["target"].to_numpy(dtype=np.int64)
train_idx, valid_idx = safe_train_valid_split(y)
valid_meta = train.iloc[valid_idx][["filename", "primary_label", "target"]].reset_index(drop=True)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
    batch_size=CFG.probe_batch_size,
    shuffle=True,
)
valid_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
    batch_size=CFG.probe_batch_size * 2,
    shuffle=False,
)

model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)

rare_count = int((pd.Series(y).value_counts() < 2).sum())
print(f"Probe train rows: {len(train_idx):,}")
print(f"Probe valid rows: {len(valid_idx):,}")
print(f"Classes with fewer than 2 rows kept in train only: {rare_count:,}")


## 4. Training And Diagnostics

Fit the probe, keep the best validation checkpoint, tune a global temperature, and write row-level, per-class, and weak-label diagnostics.


In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    """Evaluate probe loss and top-1 accuracy."""
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += xb.size(0)
    return total_loss / max(seen, 1), correct / max(seen, 1)


@torch.no_grad()
def collect_validation_logits() -> tuple[np.ndarray, np.ndarray]:
    """Collect validation logits and target indices from the best probe."""
    model.eval()
    logits_chunks, target_chunks = [], []
    for xb, yb in valid_loader:
        xb = xb.to(device)
        logits_chunks.append(model(xb).cpu().numpy())
        target_chunks.append(yb.numpy())
    return np.concatenate(logits_chunks), np.concatenate(target_chunks)


def softmax_numpy(logits: np.ndarray) -> np.ndarray:
    """Compute a stable NumPy softmax."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def negative_log_loss(logits: np.ndarray, targets: np.ndarray) -> float:
    """Compute multiclass negative log loss from logits."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return float(-log_probs[np.arange(len(targets)), targets].mean())


def fit_temperature(logits: np.ndarray, targets: np.ndarray) -> dict[str, float]:
    """Tune a global temperature on validation logits."""
    rows = []
    for temp in CFG.temperature_grid:
        rows.append(
            {
                "temperature": float(temp),
                "valid_log_loss": negative_log_loss(logits / float(temp), targets),
            }
        )
    calibration_df = pd.DataFrame(rows).sort_values("valid_log_loss")
    calibration_df.to_csv(CFG.artifact_dir / "temperature_grid.csv", index=False)
    best = calibration_df.iloc[0].to_dict()
    return {
        "temperature": float(best["temperature"]),
        "valid_log_loss": float(best["valid_log_loss"]),
        "uncalibrated_valid_log_loss": negative_log_loss(logits, targets),
    }


def collect_validation_predictions_from_logits(
    logits: np.ndarray,
    targets: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    """Create row-level and per-class validation diagnostics."""
    probs = softmax_numpy(logits)
    k = min(5, probs.shape[1])
    top_indices = np.argsort(-probs, axis=1)[:, :k]
    top_values = np.take_along_axis(probs, top_indices, axis=1)

    pred_df = valid_meta.copy()
    pred_df["top1_label"] = [idx_to_label[int(idx)] for idx in top_indices[:, 0]]
    pred_df["top1_prob"] = top_values[:, 0]
    pred_df["top5_labels"] = [
        " ".join(idx_to_label[int(idx)] for idx in row) for row in top_indices
    ]
    pred_df["top1_correct"] = pred_df["primary_label"].eq(pred_df["top1_label"]).astype(int)
    pred_df["top5_correct"] = [
        int(int(target) in row) for target, row in zip(targets, top_indices)
    ]

    class_df = (
        pred_df.groupby("primary_label")
        .agg(
            support=("filename", "count"),
            top1_recall=("top1_correct", "mean"),
            top5_recall=("top5_correct", "mean"),
            mean_top1_prob=("top1_prob", "mean"),
        )
        .reset_index()
    )
    class_df = class_df.merge(label_metadata, on="primary_label", how="left")
    class_df = class_df.sort_values(
        ["top1_recall", "top5_recall", "support"],
        ascending=[True, True, False],
    )
    return pred_df, class_df, float(pred_df["top5_correct"].mean())


def summarize_weak_labels(
    pred_df: pd.DataFrame,
    class_df: pd.DataFrame,
) -> pd.DataFrame:
    """Rank labels that need calibration, confusion, or data work."""
    error_df = pred_df[pred_df["top1_correct"].eq(0)]
    if len(error_df):
        error_summary = (
            error_df.groupby("primary_label")
            .agg(
                high_conf_errors=("top1_prob", lambda s: int((s >= 0.50).sum())),
                mean_error_confidence=("top1_prob", "mean"),
                most_common_wrong_label=("top1_label", lambda s: s.value_counts().index[0]),
            )
            .reset_index()
        )
    else:
        error_summary = pd.DataFrame(columns=["primary_label"])
    weak = class_df.merge(error_summary, on="primary_label", how="left")
    weak["high_conf_errors"] = weak["high_conf_errors"].fillna(0).astype(int)
    weak["mean_error_confidence"] = weak["mean_error_confidence"].fillna(0.0)
    weak["most_common_wrong_label"] = weak["most_common_wrong_label"].fillna("")
    weak["weak_label_score"] = (
        (1.0 - weak["top1_recall"])
        + 0.5 * (1.0 - weak["top5_recall"])
        + 0.25 * weak["mean_error_confidence"]
    )
    return weak.sort_values(
        ["weak_label_score", "support"],
        ascending=[False, False],
    )


history = []
best_acc = 0.0
epochs_without_improvement = 0

for epoch in range(1, CFG.probe_epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    valid_loss, valid_acc = evaluate(valid_loader)
    row = {
        "epoch": epoch,
        "train_loss": total_loss / max(len(train_loader.dataset), 1),
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
    }
    history.append(row)
    print(row)

    improved = valid_acc > best_acc + CFG.early_stopping_min_delta
    if improved:
        best_acc = valid_acc
        epochs_without_improvement = 0
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_perch_probe.pt",
        )
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.early_stopping_patience:
            print(f"Early stopping after {epoch} epochs; best valid accuracy: {best_acc:.4f}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)

best_checkpoint = torch_load_checkpoint(CFG.artifact_dir / "best_perch_probe.pt")
model.load_state_dict(best_checkpoint["model"])
valid_logits, valid_targets = collect_validation_logits()
validation_predictions, per_class_metrics, valid_top5_acc = collect_validation_predictions_from_logits(
    valid_logits,
    valid_targets,
)
calibration = fit_temperature(valid_logits, valid_targets)
weak_label_diagnostics = summarize_weak_labels(validation_predictions, per_class_metrics)

validation_predictions.to_csv(CFG.artifact_dir / "validation_predictions.csv", index=False)
per_class_metrics.to_csv(CFG.artifact_dir / "per_class_validation_metrics.csv", index=False)
weak_label_diagnostics.to_csv(CFG.artifact_dir / "weak_label_diagnostics.csv", index=False)
(CFG.artifact_dir / "calibration.json").write_text(
    json.dumps(calibration, indent=2),
    encoding="utf-8",
)
summary = {
    "best_valid_acc": float(best_acc),
    "valid_top5_acc": valid_top5_acc,
    "valid_log_loss": negative_log_loss(valid_logits, valid_targets),
    "calibrated_valid_log_loss": calibration["valid_log_loss"],
    "temperature": calibration["temperature"],
    "epochs_ran": len(history_df),
    "best_epoch": int(history_df.loc[history_df["valid_acc"].idxmax(), "epoch"]),
    "embedding_shape": list(embeddings.shape),
}
(CFG.artifact_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Validation top-5 accuracy: {valid_top5_acc:.4f}")
print(
    "Temperature calibration: "
    f"T={calibration['temperature']:.3f}, "
    f"log loss {calibration['uncalibrated_valid_log_loss']:.4f} -> "
    f"{calibration['valid_log_loss']:.4f}"
)
print(f"Saved outputs to {CFG.artifact_dir}")
display(weak_label_diagnostics.head(15))


## 5. Soundscape Priors

Build lightweight prior artifacts from deduplicated labeled soundscape windows. These are saved for controlled tests in the submission notebook, where priors remain disabled by default.


In [ ]:
def split_soundscape_labels(value: object) -> list[str]:
    """Split semicolon-delimited soundscape labels."""
    if pd.isna(value):
        return []
    return [label.strip() for label in str(value).split(";") if label.strip()]


def parse_soundscape_context(value: object) -> dict[str, object]:
    """Extract site and hour from a soundscape filename or row id."""
    stem = Path(str(value)).stem
    stem = re.sub(r"_\d+(?:\.0)?$", "", stem)
    parts = stem.split("_")
    site = next((part for part in parts if re.fullmatch(r"S\d+", part)), "unknown")
    time_part = next((part for part in reversed(parts) if re.fullmatch(r"\d{6}", part)), None)
    hour = int(time_part[:2]) if time_part else -1
    return {"site": site, "hour": hour}


def infer_segment_key(frame: pd.DataFrame) -> pd.Series:
    """Create a best-effort soundscape segment key from available columns."""
    if "row_id" in frame.columns:
        return frame["row_id"].astype(str)
    key = frame["filename"].astype(str)
    for time_col in ["seconds", "end_time", "end_seconds", "start_second", "start_seconds"]:
        if time_col in frame.columns:
            return key + "_" + frame[time_col].astype(str)
    return key + "_" + frame.groupby("filename").cumcount().astype(str)


def logit(values: np.ndarray) -> np.ndarray:
    """Convert probabilities to logits with clipping."""
    values = np.clip(values, 1e-5, 1 - 1e-5)
    return np.log(values / (1 - values))


def build_prior_table(
    expanded: pd.DataFrame,
    context_col: str,
    global_probs: np.ndarray,
) -> dict[str, list[float]]:
    """Build smoothed context-specific logit offsets."""
    table = {}
    for context, group in expanded.groupby(context_col):
        counts = group["label"].value_counts()
        total = max(group["segment_key"].nunique(), 1)
        probs = np.full(len(labels), CFG.prior_smoothing, dtype=np.float32)
        for label, count in counts.items():
            if label in label_to_idx:
                probs[label_to_idx[label]] += float(count)
        probs = probs / (total + 2 * CFG.prior_smoothing)
        offsets = np.clip(logit(probs) - logit(global_probs), -CFG.prior_clip, CFG.prior_clip)
        table[str(context)] = offsets.astype(float).tolist()
    return table


def build_soundscape_prior_artifact() -> dict[str, object] | None:
    """Create prior tables from train_soundscapes_labels.csv when available."""
    path = CFG.data_root / "train_soundscapes_labels.csv"
    if not path.exists():
        print("No train_soundscapes_labels.csv found; priors disabled.")
        return None

    sc = pd.read_csv(path).drop_duplicates().reset_index(drop=True)
    if not {"filename", "primary_label"}.issubset(sc.columns):
        print("Soundscape label schema is missing filename/primary_label; priors disabled.")
        return None

    context = sc["filename"].map(parse_soundscape_context).apply(pd.Series)
    sc = pd.concat([sc, context], axis=1)
    sc["segment_key"] = infer_segment_key(sc)
    sc["label_list"] = sc["primary_label"].map(split_soundscape_labels)
    expanded = sc.explode("label_list").rename(columns={"label_list": "label"})
    expanded = expanded[expanded["label"].isin(label_to_idx)].copy()
    if expanded.empty:
        print("No soundscape labels overlap the model label set; priors disabled.")
        return None

    segment_count = max(sc["segment_key"].nunique(), 1)
    global_counts = expanded["label"].value_counts()
    global_probs = np.full(len(labels), CFG.prior_smoothing, dtype=np.float32)
    for label, count in global_counts.items():
        global_probs[label_to_idx[label]] += float(count)
    global_probs = global_probs / (segment_count + 2 * CFG.prior_smoothing)

    pair_counts = {}
    for label_list in sc["label_list"]:
        clean = sorted({label for label in label_list if label in label_to_idx})
        for left, right in combinations(clean, 2):
            pair_counts.setdefault(left, {})[right] = pair_counts.setdefault(left, {}).get(right, 0) + 1
            pair_counts.setdefault(right, {})[left] = pair_counts.setdefault(right, {}).get(left, 0) + 1

    artifact = {
        "labels": labels,
        "global_logit_offset": np.clip(logit(global_probs), -CFG.prior_clip, CFG.prior_clip).astype(float).tolist(),
        "hour_offsets": build_prior_table(expanded, "hour", global_probs),
        "site_offsets": build_prior_table(expanded, "site", global_probs),
        "cooccurrence_counts": pair_counts,
        "summary": {
            "soundscape_rows": int(len(sc)),
            "deduplicated_segments": int(segment_count),
            "expanded_label_rows": int(len(expanded)),
            "overlap_labels": int(expanded["label"].nunique()),
        },
    }
    path = CFG.artifact_dir / "soundscape_priors.json"
    path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")

    summary_df = pd.Series(artifact["summary"]).to_frame("value")
    summary_df.to_csv(CFG.artifact_dir / "soundscape_prior_summary.csv")
    print(f"Saved soundscape priors to {path}")
    display(summary_df)
    return artifact


soundscape_priors = build_soundscape_prior_artifact()


## 6. Package Artifact

Package the probe checkpoint, label map, calibration, diagnostics, and soundscape-prior files for upload as a new Kaggle model version.


In [ ]:
import zipfile

zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
if FileLink is not None:
    display(FileLink(zip_path))
